# Generate IDRs and redesign a protein region

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rotskoff-group/idiom/blob/main/cookbook/notebooks/generate_sequences.ipynb)

Generate standalone IDRs and replacements between fixed protein flanks.
Outputs: candidate tables, length plots, isolated-IDR FASTA, and redesigned full-protein FASTA.

Run cells from top to bottom. In Colab select **Runtime → Change runtime type → GPU**.
A GPU is recommended; CPU works but is slower. Runtime and peak memory depend on sequence
length, model, and hardware; timings are printed below rather than promising a fixed runtime.
First use downloads model weights. Outputs are written under `OUT_DIR`; rerunning replaces
files with the same names. Download that folder from Colab before ending the session.


In [ ]:
import importlib.util
import subprocess
import sys
if importlib.util.find_spec("idiom") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/rotskoff-group/idiom.git@v1"])

if importlib.util.find_spec("pandas") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas"])

import json
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from idiom import IDiom, IDiomSAE
from idiom.data.records import Record, read_fasta, parse_idr_header
print("Python:", sys.version.split()[0])
import idiom
print("IDiom:", idiom.__file__)


# Locate the companion helper in a clone, or download it for standalone Colab use.
helper_dir = next((p for p in (Path.cwd(), Path.cwd() / "cookbook/notebooks")
                   if (p / "workflow_utils.py").is_file()), None)
if helper_dir is None:
    from urllib.request import urlretrieve
    helper_dir = Path(".idiom_notebook_helpers")
    helper_dir.mkdir(exist_ok=True)
    urlretrieve("https://raw.githubusercontent.com/rotskoff-group/idiom/main/"
                "cookbook/notebooks/workflow_utils.py", helper_dir / "workflow_utils.py")
sys.path.insert(0, str(helper_dir.resolve()))
from workflow_utils import (AA, DEMO, load_inputs, idr_sequence, isolated, check_context,
                            summaries, write_fasta, save_run)


In [ ]:
MODEL = "jxliu2/idiom-20M"
DEVICE = "auto"
N = 8
BATCH_SIZE = 4
SEED = 0
TEMPERATURE = 1.0
TOP_P = 0.95
LENGTH_RANGE = (20, 60)
MAX_NEW_TOKENS = 128
OUT_DIR = Path("generation_outputs")
# Optional: annotated full-protein FASTA; first accepted record is redesigned.
INPUT_FASTA = None
INPUT_MODE = "annotated"
MAX_RECORDS = 1


In [ ]:
started = time.perf_counter()


## Generate standalone IDRs

`length_range` filters oversampled candidates, so fewer than `N` may be returned. This is not
fixed-length generation. A seed is reproducible for fixed settings including batch size.
These candidates require downstream evaluation; generation alone does not establish disorder
or function. Lower `BATCH_SIZE` if GPU memory is limited.


In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
model = IDiom.from_pretrained(MODEL, device=DEVICE)
kwargs = dict(n=N, batch_size=BATCH_SIZE, seed=SEED, temperature=TEMPERATURE,
              top_p=TOP_P, length_range=LENGTH_RANGE, max_new_tokens=MAX_NEW_TOKENS)
idrs = model.generate_unprompted(**kwargs)
generated = [Record(f"generated_{i}", s, 0, len(s)) for i, s in enumerate(idrs) if s]
write_fasta(generated, OUT_DIR / "idrs.fasta")
print(f"Requested {N}; retained {len(generated)} nonempty IDRs")
display(pd.DataFrame([dict(record_id=r.accession, sequence=r.full_seq, length=len(r.full_seq)) for r in generated]))


## Redesign an annotated region

The default is the bundled HP1α example (IDR residues 79–123). For your own protein, upload an
annotated FASTA and set `INPUT_FASTA`. Flanks are retained unchanged; generated IDRs can have
different lengths, so the exported coordinates are updated. The original IDR is saved as a
reference. Only the first valid record is used here; the package's `generate_prompted_fasta`
method supports batch workflows.


In [ ]:
from huggingface_hub import hf_hub_download
prompt_path = INPUT_FASTA
if prompt_path is None:
    prompt_path = hf_hub_download("jxliu2/idiom-db", "other/example_data/prompted_grpo/P45973.fasta",
                                 repo_type="dataset")
records, audit = load_inputs(prompt_path, INPUT_MODE, MAX_RECORDS)
audit.to_csv(OUT_DIR / "input_audit.csv", index=False)
display(audit)
if not records:
    raise ValueError("No accepted annotated proteins.")
record = records[0]
check_context(records, model.model.cfg.max_seq_len, include_flanks=True)
print("Original IDR:", idr_sequence(record))
replacement = model.generate_prompted(record.full_seq, record.idr_start, record.idr_end, **kwargs)
redesigned = []
for i, s in enumerate(replacement):
    if s:
        full = record.full_seq[:record.idr_start] + s + record.full_seq[record.idr_end:]
        redesigned.append(Record(f"{record.accession}_design_{i}", full, record.idr_start, record.idr_start + len(s)))
write_fasta(redesigned, OUT_DIR / "redesigned_proteins.fasta")
write_fasta(isolated(redesigned), OUT_DIR / "redesigned_idrs.fasta")
write_fasta(isolated(records[:1]), OUT_DIR / "original_idr.fasta")
print(f"Retained {len(redesigned)} prompted replacements")


## Inspect and export candidates

The table connects each prompted design to its input record. Compare sequence length and
composition first; use the comparison notebook for embeddings, SAE features, and optional
aggregate likelihood scoring. Empty outputs remain valid files but cannot be analyzed downstream.


In [ ]:
rows = [dict(record_id=r.accession, source_record=None, kind="unprompted", sequence=idr_sequence(r),
             length=len(idr_sequence(r))) for r in generated]
rows += [dict(record_id=r.accession, source_record=record.accession, kind="prompted",
              sequence=idr_sequence(r), length=len(idr_sequence(r))) for r in redesigned]
candidates = pd.DataFrame(rows, columns=["record_id", "source_record", "kind", "sequence", "length"])
candidates.to_csv(OUT_DIR / "candidates.csv", index=False)
display(candidates)
if len(candidates):
    fig, ax = plt.subplots(figsize=(6, 3), constrained_layout=True)
    for label, group in candidates.groupby("kind"):
        ax.hist(group.length, bins=10, alpha=0.5, label=label)
    ax.set(xlabel="IDR length", ylabel="Candidate count")
    ax.legend()
    fig.savefig(OUT_DIR / "candidate_lengths.png", dpi=160)
    plt.show()
save_run(OUT_DIR, dict(model=MODEL, device=str(model.device), input=prompt_path, **kwargs), elapsed=time.perf_counter() - started)
print(f"Elapsed including model load: {time.perf_counter() - started:.1f} s")


## Use the outputs

Keep `input_audit.csv` with your results: `record_id` connects exported rows to the original
accession and IDR span, even when accessions repeat. `run.json` records the settings and installed
package versions. Record exact model revisions separately when freezing a published analysis.

Next: open [compare_sequence_sets.ipynb](compare_sequence_sets.ipynb) with
`redesigned_idrs.fasta` as the query set and your natural IDRs as the reference set.
